# Episode 6: Profiling a GPT with PyTorch's profiler

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep006-profiling.ipynb)

We look at our model as a graph of operations in Model Explorer, profile it with PyTorch's profiler, and measure how decisions in its code change its speed on a CPU and a GPU.

[The video](https://youtu.be/7C0USj9KtI0) · [The episode's page](https://mvgpt.nenuadrian.com/ep006-profiling/) · [The docs](https://nenuadrian.github.io/mvGPT/profiling/) · [The code at `ep006/end`](https://github.com/nenuadrian/mvGPT/tree/ep006/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep006/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("sft")

## The model as a graph
`explore.py` exports the model with torch.export and opens it in Model Explorer. It runs in
the notebook itself, so Model Explorer shows inside the cell.

In [ ]:
%run explore.py checkpoints/sft/model.pt

## Where the time goes
PyTorch's profiler on the CPU and on the GPU. The episode's GPU was a laptop's
(`--device mps`); here it's Colab's, so the profiler sees its CUDA kernels too.

In [ ]:
!python profiler.py checkpoints/sft/model.pt --device cpu --out results 2>/dev/null
!python profiler.py checkpoints/sft/model.pt --device cuda --out results 2>/dev/null

The graph again, each operation coloured by its time on the CPU:

In [ ]:
%run explore.py checkpoints/sft/model.pt --times

## Decisions in the code, measured

In [ ]:
!python decisions.py checkpoints/sft/model.pt --device cpu --out results 2>/dev/null
!python decisions.py checkpoints/sft/model.pt --device cuda --out results 2>/dev/null

Next: [episode 7, DPO: teaching a GPT from preferences](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep007-dpo.ipynb)